# Testing Navier Stokes solver.

1. Convert twoD -> threeD. DONE IN `TestingNavierStokes1.ipynb`
2. Add hydrostatic balance. DONE IN `TestingNavierStokes2.ipynb`
3. Add Coriolis force and customize to wind_forced_dishpan problem, except that the surface boundary condition is Dirichlet with a specified surface velocity. DONE IN `TestingNavierStokes3.ipynb`
4. Customize to stress boundary condition at upper surface, not specified velocity. Add Jacobian to solver.
4b. Add linear solve too, to test performance. Replace total pressure with pressure anomaly.
5. Switch to preconditioned FGMRES solver

Based on: `gridap` Tutorial 8: Incompressible Navier-Stokes

twnh July '25

## Problem statement

The ultimate goal is to solve a nonlinear multi-field PDE. Consider here the lid-stress-driven flow for the incompressible rotating Navier-Stokes equations. Formally, the PDE we want to solve is: find the velocity vector $u$ and the pressure anomaly $p$ such that

$$
\left\lbrace
\begin{aligned}
-\nu  \nabla^2 u +  \frac{1}{\rho_0} \nabla p + f  \hat{\mathbf k} \times u = 0 &\text{ in }\Omega,\\
\nabla\cdot u = 0 &\text{ in } \Omega,\\
\boldsymbol{t} \cdot \boldsymbol{\sigma} \cdot \mathbf{n} = \tau_{\text{imposed}} &\text{ on } \Gamma_s, \\
u = 0 &\text{ on } \Gamma_w,
\end{aligned}
\right.
$$

where the computational domain is the rectangle $\Omega \doteq (0,L_x) \times (-L_y/2,L_y/2) \times (-L_z,0)$, ${\mathbf n}$ is the unit outward normal, and $\boldsymbol{t}$ is the tangential direction at the surface, $\Gamma_s$. The driving force is the tangential stress $\tau_{\text{imposed}}$ (units of $\text{m}^{2} \text{s}^{-2}$ ). The mean value of the pressure anomaly is constrained to equal zero,

$$
\int_\Omega p \ {\rm d}\Omega = 0, 
$$
and the total pressure is
$$
p_{\text{tot}} = p - \rho_0 g_r z .
$$

The weak form of this problem is (from GPT-4.1):

Find $(u,p,\lambda) \in V \times Q \times \mathbb{R}$ such that **for all** $(v, q, \mu) \in V \times Q \times \mathbb{R}$:

$$
\begin{aligned}
&\int_\Omega \nu \nabla u : \nabla v \, d\Omega
+ \int_\Omega (f\, \hat{\mathbf{k}} \times u) \cdot v\, d\Omega
- \frac{1}{\rho_0} \int_\Omega p\, \nabla \cdot v\, d\Omega 
+ \frac{1}{\rho_0} \int_\Omega q\, \nabla \cdot u\, d\Omega
+ \frac{1}{\rho_0} \int_\Omega \lambda\, q\, d\Omega
+ \frac{1}{\rho_0} \int_\Omega \mu\, p\, d\Omega 
= \int_{\Gamma_s} \tau_{\text{imposed}}\, (t \cdot v)\, dS
\end{aligned}
$$

Alternatively, by restricting the $p$ and $q$ fields to have zero mean by definition of their functions spaces, the weak form is:

$$
\boxed{
\begin{aligned}
&\int_\Omega \nu \nabla u : \nabla v \, d\Omega
+ \int_\Omega (f\, \hat{\mathbf{k}} \times u) \cdot v\, d\Omega
- \frac{1}{\rho_0} \int_\Omega p\, \nabla \cdot v\, d\Omega 
+ \frac{1}{\rho_0} \int_\Omega q\, \nabla \cdot u\, d\Omega
= \int_{\Gamma_s} \tau_{\text{imposed}}\, (t \cdot v)\, dS
\end{aligned}
}
$$

In [1]:
using Gridap
using GridapSolvers
using GridapSolvers.LinearSolvers, GridapSolvers.MultilevelTools, GridapSolvers.NonlinearSolvers
using GridapSolvers.BlockSolvers: LinearSystemBlock, NonlinearSystemBlock, BiformBlock, BlockTriangularSolver
using Gridap.MultiField
using LinearAlgebra

#### Define parameters

In [2]:
# The grid
Nx = 32             # number of points in x direction
Ny = 16             # number of points in y direction
Nz = 8            # number of points in the vertical direction

# The domain
Lx = 0.25            # (m) domain length
Ly = 0.125           # (m) domain width
Lz = 0.025           # (m) domain depth
domain = (0.0,Lx,-Ly/2,Ly/2,-Lz,0.0)
partition = (Nx,Ny,Nz)

# Create the mesh
model = CartesianDiscreteModel(domain,partition)

# Physical properties
# ν = 1e-6           # (m^2/s) kinematic viscosity
# ρₒ = 1000.0       # (kg/m^3) reference density
ν = 1.0e-3           # (m^2/s) kinematic viscosity
ρₒ = 1000.0       # (kg/m^3) reference density
# Rotation rate
rotation_period = 6.0   # (s)
f = 2*2*π/rotation_period
# f=0.0
Ekman_layer_depth = sqrt(2*ν / f) # (m), Ekman layer depth
println("Ekman layer depth: ", Ekman_layer_depth, " m")

# Surface stress boundary condition
u₁₀(y) = 0.25     # m s⁻¹, average wind velocity 10 meters above the ocean
# u₁₀(y) = -0.25 + 0.25 .* cos(π.*y./Ly)     # m s⁻¹, average wind velocity 10 meters above the ocean
# u₁₀(y) =  0.25 .* cos(π.*y./Ly)     # m s⁻¹, average wind velocity 10 meters above the ocean
cᴰ = 2.5e-3 # dimensionless drag coefficient
ρₐ = 1.225  # kg m⁻³, average density of air at sea-level
Qᵘ(x) = VectorValue((ρₐ / ρₒ) * cᴰ * u₁₀(x[2]) * abs(u₁₀(x[2])),0,0) # m² s⁻²

Ekman layer depth: 0.03090193616185517 m


Qᵘ (generic function with 1 method)

Create two new boundary tags,  namely `"lid"` and `"walls"` for the top side of the square (where the stress is applied), and for the rest of the boundary (where the velocity is zero).

In [3]:
labels = get_face_labeling(model)
add_tag_from_tags!(labels,"lid",[22,])
add_tag_from_tags!(labels,"walls",[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,23,24,25,26]) ;

## FE spaces

For the velocities, create a conventional vector-valued continuous Lagrangian FE space. In this example, we select a second order interpolation.

In [4]:
order = 2
qdegree = 2*(order+1)
reffeᵤ = ReferenceFE(lagrangian,VectorValue{3,Float64},order)
V = TestFESpace(model,reffeᵤ,conformity=:H1,labels=labels,dirichlet_tags="walls")

UnconstrainedFESpace()

The interpolation space for the pressure anomaly is built as follows

In [5]:
reffeₚ = ReferenceFE(lagrangian,Float64,order-1;space=:P)
Q = TestFESpace(model,reffeₚ,conformity=:L2,constraint=:zeromean)

ZeroMeanFESpace()

With the options `:Lagrangian`, `space=:P`, `valuetype=Float64`, and `order=order-1`, we select the local polynomial space $P_{k-1}(T)$ on the cells $T\in\mathcal{T}$. With the symbol `space=:P` we specifically chose a local Lagrangian interpolation of type "P". Without using `space=:P`, would lead to a local Lagrangian of type "Q" since this is the default for quadrilateral or hexahedral elements. On the other hand, `constraint=:zeromean` leads to a FE space, whose functions are constrained to have mean value equal to zero, which is just what we need for the pressure anomaly space. With these objects, we build the trial multi-field FE spaces

In [6]:
uD0 = VectorValue(0,0,0)            # Velocity vanishes on the walls
U = TrialFESpace(V,uD0)
P = TrialFESpace(Q)
mfs = Gridap.MultiField.BlockMultiFieldStyle()
Y = MultiFieldFESpace([V, Q];style=mfs)
X = MultiFieldFESpace([U, P];style=mfs)

MultiFieldFESpace()

## Triangulation and integration quadrature

From the discrete model we can define the triangulation and integration measure

In [7]:
degree = order
Ω = Triangulation(model)
dΩ = Measure(Ω,qdegree)
Γ = BoundaryTriangulation(model,tags="lid")
dΓ = Measure(Γ,degree)

GenericMeasure()

## Preconditioned FGMRES solver.

See: https://gridap.github.io/GridapSolvers.jl/stable/Examples/NavierStokes/

In [8]:
khat = VectorValue(0,0,1)
Coriolis(u,v,dΩ) = ∫(f * cross(khat, u) ⋅ v)dΩ # Coriolis term
b((v,q),dΓ) =  ∫(- v ⋅ Qᵘ)dΓ    # Boundary condition for the velocity

α = 1.e2    # Stabilization parameter
Π_Qh = LocalProjectionMap(divergence,Q,qdegree)
graddiv(u,v,dΩ) = ∫(α*(∇⋅v)⋅Π_Qh(u))dΩ

# conv(u,∇u) = (∇u')⋅u
# dconv(du,∇du,u,∇u) = conv(u,∇du)+conv(du,∇u)
# c(u,v,dΩ) = ∫(v⊙(conv∘(u,∇(u))))dΩ
# dc(u,du,dv,dΩ) = ∫(dv⊙(dconv∘(du,∇(du),u,∇(u))))dΩ

lap(u,v,dΩ) = ∫(ν*∇(v)⊙∇(u))dΩ

# jac_u(u,du,dv,dΩ) = lap(du,dv,dΩ) + dc(u,du,dv,dΩ) + graddiv(du,dv,dΩ)
jac_u(u,du,dv,dΩ) = lap(du,dv,dΩ) + graddiv(du,dv,dΩ) + Coriolis(du,dv,dΩ)
# jac_u(u,du,dv,dΩ) = lap(du,dv,dΩ) + graddiv(du,dv,dΩ)
jac((u,p),(du,dp),(dv,dq),dΩ) = jac_u(u,du,dv,dΩ) - ∫(divergence(dv)*dp)dΩ - ∫(divergence(du)*dq)dΩ

# res_u(u,v,dΩ) = lap(u,v,dΩ) + c(u,v,dΩ) + graddiv(u,v,dΩ)
res_u(u,v,dΩ) = lap(u,v,dΩ) + graddiv(u,v,dΩ) + Coriolis(u,v,dΩ)
# res_u(u,v,dΩ) = lap(u,v,dΩ) + graddiv(u,v,dΩ)
res((u,p),(v,q),dΩ) = res_u(u,v,dΩ) - ∫(divergence(v)*p)dΩ - ∫(divergence(u)*q)dΩ + b((v,q),dΓ)

jac_h(x,dx,dy) = jac(x,dx,dy,dΩ)
res_h(x,dy) = res(x,dy,dΩ)
op = FEOperator(res_h,jac_h,X,Y)

solver_u = LUSolver()
solver_p = CGSolver(JacobiLinearSolver();maxiter=20,atol=1e-14,rtol=1.e-6,verbose=true)
solver_p.log.depth = 4

bblocks  = [NonlinearSystemBlock() LinearSystemBlock();
            LinearSystemBlock()    BiformBlock((p,q) -> ∫(-(1.0/α)*p*q)dΩ,Q,Q)]
coeffs = [1.0 1.0;
          0.0 1.0]  
P = BlockTriangularSolver(bblocks,[solver_u,solver_p],coeffs,:upper)
solver = FGMRESSolver(20,P;atol=1e-11,rtol=1.e-8,verbose=true)
solver.log.depth = 2

nlsolver = NewtonSolver(solver;maxiter=20,atol=1e-10,rtol=1.e-12,verbose=true)
uh,ph = solve(nlsolver,op)

writevtk(Ω,"TestingNavierStokes5",cellfields=["uh"=>uh,"ph"=>ph])

--------------- Starting Newton-Raphson solver --------
  > Iteration   0 - Residuals: 1.45e-10,   1.00e+00 
    --------------- Starting FGMRES solver ----------------
      > Iteration   0 - Residuals: 1.45e-10,   1.00e+00 
        --------------- Starting CG solver --------------------
          > Iteration   0 - Residuals: 0.00e+00,   1.00e+00 
        Solver CG finished with reason SOLVER_CONVERGED_ATOL
        Iterations:   0 - Residuals: 0.00e+00,   NaN 
        --------------- Exiting CG solver ---------------------
      > Iteration   1 - Residuals: 7.51e-15,   5.16e-05 
    Solver FGMRES finished with reason SOLVER_CONVERGED_ATOL
    Iterations:   1 - Residuals: 7.51e-15,   5.16e-05 
    --------------- Exiting FGMRES solver -----------------
  > Iteration   1 - Residuals: 7.51e-15,   5.16e-05 
Solver Newton-Raphson finished with reason SOLVER_CONVERGED_ATOL
Iterations:   1 - Residuals: 7.51e-15,   5.16e-05 
--------------- Exiting Newton-Raphson solver ---------


(["TestingNavierStokes5.vtu"],)